In [3]:
# ---- class_weight comparison: does weighting the minority class reduce false negatives? ----
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report, recall_score
import pandas as pd

In [4]:
# Rebuild X, y (Amyloid + Total tau, MCI only) — same as cell 45
df = pd.read_csv("data/plasma_lipidomics.csv")

In [5]:
mci = df[df['Diagnostic'] == 'Mild Cognitive Impairment'].copy()
amyloid_median = mci['CSF Amyloid (pg/mL)'].median()
mci['CSF Amyloid (pg/mL)'] = mci['CSF Amyloid (pg/mL)'].fillna(amyloid_median)
tau_median = mci['CSF Total tau (pg/mL)'].median()
mci['CSF Total tau (pg/mL)'] = mci['CSF Total tau (pg/mL)'].fillna(tau_median)
mci['target'] = (mci["Progression to Alzheimer's Disease"] == 'Yes').astype(int)
y = mci['target']
X = mci[['CSF Amyloid (pg/mL)', 'CSF Total tau (pg/mL)']]

print("Class balance (0=No progression, 1=Progressed):")
print(y.value_counts(), "\n")

Class balance (0=No progression, 1=Progressed):
target
1    47
0    42
Name: count, dtype: int64 



In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

weight_options = {
    'None (baseline)':           None,
    "'balanced'":                'balanced',
    "custom {0:1, 1:2}":         {0: 1, 1: 2},
    "custom {0:1, 1:3}":         {0: 1, 1: 3},
    "custom {0:1, 1:4}":         {0: 1, 1: 4},
    "custom {0:1, 1:5}":         {0: 1, 1: 5},
    "custom {0:1, 1:6}":         {0: 1, 1: 6},
}

for label, cw in weight_options.items():
    model = Pipeline([
        ('sc', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight=cw))
    ])
    y_pred = cross_val_predict(model, X, y, cv=cv, method='predict')

    print(f"--- class_weight = {label} ---")
    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    print(pd.DataFrame(cm, index=['Actual: No', 'Actual: Yes'], columns=['Pred: No', 'Pred: Yes']))
    print(classification_report(y, y_pred, target_names=['No progression', 'Progressed'], digits=3))
    print('Recall (Progressed):', recall_score(y, y_pred))
    print("\n\n-------------------------------------\n")

--- class_weight = None (baseline) ---
             Pred: No  Pred: Yes
Actual: No         29         13
Actual: Yes        11         36
                precision    recall  f1-score   support

No progression      0.725     0.690     0.707        42
    Progressed      0.735     0.766     0.750        47

      accuracy                          0.730        89
     macro avg      0.730     0.728     0.729        89
  weighted avg      0.730     0.730     0.730        89

Recall (Progressed): 0.7659574468085106


-------------------------------------

--- class_weight = 'balanced' ---
             Pred: No  Pred: Yes
Actual: No         30         12
Actual: Yes        11         36
                precision    recall  f1-score   support

No progression      0.732     0.714     0.723        42
    Progressed      0.750     0.766     0.758        47

      accuracy                          0.742        89
     macro avg      0.741     0.740     0.740        89
  weighted avg      0.741  

In [12]:
# ---- class_weight sweep: confusion matrix + ROC-AUC + PR-AUC side by side ----
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (confusion_matrix, classification_report,
                              recall_score, precision_score,
                              roc_auc_score, average_precision_score)


weight_options = {
    'None (baseline)':     None,
    "'balanced'":           'balanced',
    "custom {0:1, 1:2}":    {0: 1, 1: 2},
    "custom {0:1, 1:3}":    {0: 1, 1: 3},
    "custom {0:1, 1:6}":    {0: 1, 1: 6},
}

summary_rows = []

for label, cw in weight_options.items():
    model = Pipeline([
        ('sc', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight=cw))
    ])
    y_pred  = cross_val_predict(model, X, y, cv=cv, method='predict')
    y_proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]

    cm = confusion_matrix(y, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    print(f"--- class_weight = {label} ---")
    print(pd.DataFrame(cm, index=['Actual: No', 'Actual: Yes'], columns=['Pred: No', 'Pred: Yes']))
    print(classification_report(y, y_pred, target_names=['No progression', 'Progressed'], digits=3))

    roc_auc = roc_auc_score(y, y_proba)
    pr_auc  = average_precision_score(y, y_proba)   # PR-AUC, positive class = 1
    print(f"ROC-AUC: {roc_auc:.3f}   PR-AUC: {pr_auc:.3f}\n")

    summary_rows.append({
        'class_weight':   label,
        'FN': fn, 'FP': fp,
        'Recall (Progressed)':    recall_score(y, y_pred),
        'Precision (Progressed)': precision_score(y, y_pred),
        'ROC-AUC': round(roc_auc, 3),
        'PR-AUC':  round(pr_auc, 3),
    })

# ---- Summary table across all weight settings ----
summary_df = pd.DataFrame(summary_rows)
print("=== Summary across class_weight sweep ===")
print(summary_df.to_string(index=False))

--- class_weight = None (baseline) ---
             Pred: No  Pred: Yes
Actual: No         29         13
Actual: Yes        11         36
                precision    recall  f1-score   support

No progression      0.725     0.690     0.707        42
    Progressed      0.735     0.766     0.750        47

      accuracy                          0.730        89
     macro avg      0.730     0.728     0.729        89
  weighted avg      0.730     0.730     0.730        89

ROC-AUC: 0.801   PR-AUC: 0.828

--- class_weight = 'balanced' ---
             Pred: No  Pred: Yes
Actual: No         30         12
Actual: Yes        11         36
                precision    recall  f1-score   support

No progression      0.732     0.714     0.723        42
    Progressed      0.750     0.766     0.758        47

      accuracy                          0.742        89
     macro avg      0.741     0.740     0.740        89
  weighted avg      0.741     0.742     0.741        89

ROC-AUC: 0.801   PR